# ქართული RAG: `gemma3:4b` vs Gemma 4 (Ollama) — Google Colab

ეს ნოუთბუქი ერთ სესიაში აყენებს Ollama-ს GPU-ზე, იტვირთავს ორ მოდელს და ადარებს **იგივე** მინი-corpus + იგივე კითხვებზე მარტივ **TF-IDF** retrieval-ით.

**სანამ დაიწყებ:** `Runtime` → `Change runtime type` → **GPU** (T4/L4).

**მოდელის სახელები:** ქვემოთ უჯრაში შეცვალე `GEMMA4_TAG`, თუ Ollama-ში სხვა ტეგი გაქვს (მაგ. `gemma4:e2b`, `gemma4:26b`). ნაგულისხმევად: `gemma3:4b` და `gemma4:e4b`.

**VRAM:** ორი მოდელი ერთდროულად შეიძლება არ ჩაეტიოს — ბოლო უჯრა თითო მოდელზე გაშვების შემდეგ `ollama stop`-ს იძახებს.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. დაყენება: `zstd` + Ollama

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 3. `ollama serve` ფონურად

In [ ]:
import os, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

log_path = "/content/ollama.log"
log_file = open(log_path, "w")
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        print(f"Ollama მუშაობს (~{i+1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama არ ამუშავდა დროში. ლოგი:")
    print(open(log_path).read()[-2000:])

## 4. მოდელების ტეგები და pull

`GEMMA4_TAG`-ს შეცვალე თუ სხვა Gemma 4 გინდა (იხილე [ollama.com/library/gemma4](https://ollama.com/library/gemma4)).

In [ ]:
GEMMA3_TAG = "gemma3:4b"
GEMMA4_TAG = "gemma4:e4b"  # ან მაგ. gemma4:e2b, gemma4:26b — შენი VRAM-ის მიხედვით

!ollama pull {GEMMA3_TAG}
!ollama pull {GEMMA4_TAG}
!ollama list

## 5. დამოკიდებულებები (რაგი + მეტრიკა)

In [ ]:
!pip install -q scikit-learn

## 6. Corpus, კითხვები და შედარება

- **Retrieval:** `TfidfVectorizer` + cosine similarity (სწრაფი baseline; პროდაქშენში გინდა embedding-index).
- **ქართული სტაბილურობა:** პასუხში ქართული ასოების წილი (Unicode `10A0–10FF`). დაბალი წილი = ენის „გაქცევა“.
- **Grounding:** მოსალოდნელი ქვესტრიქები (must_contain) — მარტივი ავტომატური შემოწმება.

საკუთარი corpus-ისთვის შეცვალე `CHUNKS` / `TASKS` ან დაამატე უჯრა JSON-ის ატვირთვით.

In [ ]:
from __future__ import annotations

import json
import time
import urllib.request
from dataclasses import dataclass

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OLLAMA = "http://127.0.0.1:11434"

SYSTEM_RAG_GE = """You answer using ONLY the provided CONTEXT.
Rules:
- Reply ONLY in Georgian (Mkhedruli script). No English or other languages unless they appear verbatim inside CONTEXT as proper names.
- If CONTEXT does not contain the answer, reply exactly: არ მაქვს საკმარისი ინფორმაცია.
- Be concise (2–6 sentences). Do not invent facts outside CONTEXT.
"""

USER_RAG_TEMPLATE = """კონტექსტი:
{context}

კითხვა:
{question}
"""

# --- სატესტო corpus (შეცვალე შენი დომენით) ---
CHUNKS: list[str] = [
    "საქართველოს დედაქალაქია თბილისი. იგი მდებარეობს მდინარე მტკვრის ორივე ნაპირზე.",
    "ლარი არის საქართველოს ოფიციალური ვალუტა. ერთ ლარში არის 100 თეთრი.",
    "ქართული ენა ეკუთვნის ქართველურ ენათა ოჯახს და იწერება მხედრული ანბანით.",
    "სამედიცინო ჩანაწერში: პაციენტს აღენიშნება არტერიული ჰიპერტენზია; რეკომენდებულია წნევის მონიტორინგი და დიეტის კონტროლი.",
]


@dataclass
class RagTask:
    question: str
    must_contain: list[str]  # მარტივი grounding-შემოწმება


TASKS: list[RagTask] = [
    RagTask("საქართველოს დედაქალაქი რომელია?", ["თბილის"]),
    RagTask("რომელი ვალუტა მოქმედებს საქართველოში?", ["ლარ"]),
    RagTask("რომელ ანბანს იყენებს ქართული ენა?", ["მხედრულ", "მხედრული"]),
    RagTask("რა რეკომენდაციაა წნევასთან დაკავშირებით ამ ჩანაწერში?", ["მონიტორინგ", "დიეტ"]),
]


def georgian_letter_ratio(text: str) -> float:
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 1.0
    ge = sum(1 for ch in letters if "\u10a0" <= ch <= "\u10ff")
    return ge / len(letters)


def ollama_generate(model: str, system: str, prompt: str, *, temperature: float = 0.1, num_predict: int = 512) -> tuple[str, float]:
    payload = {
        "model": model,
        "system": system,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        f"{OLLAMA}/api/generate",
        data=body,
        headers={"Content-Type": "application/json; charset=utf-8"},
        method="POST",
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=300) as resp:
        raw = resp.read().decode("utf-8", errors="replace")
    dt = time.perf_counter() - t0
    data = json.loads(raw)
    return str(data.get("response", "")).strip(), dt


def retrieve_top1(question: str, chunks: list[str], vectorizer: TfidfVectorizer, X) -> tuple[int, float]:
    qv = vectorizer.transform([question])
    sims = cosine_similarity(qv, X)[0]
    i = int(np.argmax(sims))
    return i, float(sims[i])


def eval_model(model_tag: str, chunks: list[str], tasks: list[RagTask]) -> dict:
    vec = TfidfVectorizer()
    X = vec.fit_transform(chunks)
    rows = []
    latencies = []
    geo_ratios = []
    hits = 0

    for t in tasks:
        idx, sim = retrieve_top1(t.question, chunks, vec, X)
        ctx = chunks[idx]
        prompt = USER_RAG_TEMPLATE.format(context=ctx, question=t.question)
        ans, dt = ollama_generate(model_tag, SYSTEM_RAG_GE, prompt)
        latencies.append(dt)
        gr = georgian_letter_ratio(ans)
        geo_ratios.append(gr)
        ok = all(s.lower() in ans.lower() for s in t.must_contain)
        hits += int(ok)
        rows.append(
            {
                "question": t.question,
                "retrieved_idx": idx,
                "retrieval_score": round(sim, 4),
                "latency_s": round(dt, 3),
                "georgian_letter_ratio": round(gr, 3),
                "must_contain_ok": ok,
                "answer_preview": ans[:280] + ("…" if len(ans) > 280 else ""),
            }
        )

    return {
        "model": model_tag,
        "n_tasks": len(tasks),
        "must_contain_accuracy": hits / len(tasks),
        "avg_latency_s": float(np.mean(latencies)),
        "avg_georgian_letter_ratio": float(np.mean(geo_ratios)),
        "rows": rows,
    }


def stop_model(name: str) -> None:
    import subprocess

    subprocess.run(["ollama", "stop", name], capture_output=True, text=True)


print("სვეტები მზადაა. შემდეგ უჯრაში გაეშვება ორივე მოდელი თანმიმდევრულად.")

In [ ]:
import subprocess

# თანმიმდევრობა: ჯერ gemma3, stop, შემდეგ gemma4 — VRAM-ის დასაზოგად
subprocess.run(["ollama", "stop", GEMMA3_TAG], capture_output=True)
subprocess.run(["ollama", "stop", GEMMA4_TAG], capture_output=True)

r3 = eval_model(GEMMA3_TAG, CHUNKS, TASKS)
print("===", GEMMA3_TAG, "===")
print(json.dumps({k: r3[k] for k in r3 if k != "rows"}, ensure_ascii=False, indent=2))

stop_model(GEMMA3_TAG)

r4 = eval_model(GEMMA4_TAG, CHUNKS, TASKS)
print("===", GEMMA4_TAG, "===")
print(json.dumps({k: r4[k] for k in r4 if k != "rows"}, ensure_ascii=False, indent=2))

stop_model(GEMMA4_TAG)

print("\n--- შედარება ---")
print(f"must_contain: {GEMMA3_TAG}={r3['must_contain_accuracy']:.2%} vs {GEMMA4_TAG}={r4['must_contain_accuracy']:.2%}")
print(f"avg_latency_s: {GEMMA3_TAG}={r3['avg_latency_s']:.2f} vs {GEMMA4_TAG}={r4['avg_latency_s']:.2f}")
print(f"avg_georgian_ratio: {GEMMA3_TAG}={r3['avg_georgian_letter_ratio']:.3f} vs {GEMMA4_TAG}={r4['avg_georgian_letter_ratio']:.3f}")

with open("/content/rag_compare_summary.json", "w", encoding="utf-8") as f:
    json.dump({"gemma3": r3, "gemma4": r4}, f, ensure_ascii=False, indent=2)
print("\nშენახულია: /content/rag_compare_summary.json")

## 7. (სურვილისამებრ) სრული პასუხების ნახვა

In [ ]:
print(json.dumps(r3["rows"], ensure_ascii=False, indent=2))
print("---")
print(json.dumps(r4["rows"], ensure_ascii=False, indent=2))

## 8. ჩამოტვირთვა

In [ ]:
from google.colab import files

files.download("/content/rag_compare_summary.json")